# 運用① — モデルの保存・読込と推論時間の測定

出典：第06章 02節／パス：`06_ops/01_baseline.ipynb`（ベースライン）／パッケージ：numpy・pandas・scikit-learn・joblib／実行時間の目安：約1分

このノート単体で実行できる。小さな審査モデルを作り、Pipelineごと保存→読込→一致確認→速度測定までを通す。

In [1]:
import time

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RS = 0
INPUT = "input/mini_applications.csv"
try:
    _m = pd.read_csv(INPUT)
    y = _m["default"].to_numpy()
    X = _m.drop(columns=["default"])
    print(f"Inputを使用: {INPUT}")
except FileNotFoundError:
    rng = np.random.default_rng(60)
    N = 4000
    income = np.clip(rng.normal(450.0, 150.0, size=N), 150.0, 1500.0)
    past_due = np.clip(rng.poisson(0.4, size=N), 0, 5)
    use_rate = rng.uniform(0.0, 1.0, size=N)
    housing = rng.choice(["rent", "own", "company"], size=N, p=[0.55, 0.35, 0.10])
    logit = -3.4 + 0.9 * past_due + 1.2 * use_rate - 0.003 * (income - 450.0) + 0.2 * (housing == "rent")
    y = (rng.uniform(size=N) < 1.0 / (1.0 + np.exp(-logit))).astype(int)
    X = pd.DataFrame({"income": income, "past_due": past_due, "use_rate": use_rate, "housing": housing})
    print("Inputが無いため合成データを生成")
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RS)
pre = ColumnTransformer([
    ("num", StandardScaler(), ["income", "past_due", "use_rate"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["housing"]),
])
pipe = make_pipeline(pre, HistGradientBoostingClassifier(max_iter=100, random_state=RS)).fit(X_tr, y_tr)
print(f"件数: {len(X)}, 貸倒率: {pd.Series(y).mean():.3f}")


Inputを使用: input/mini_applications.csv


件数: 4000, 貸倒率: 0.108


In [2]:
# 保存→読込。前処理ごと固めるのが再現の要点（列の順序・欠損扱いがずれない）
joblib.dump(pipe, "credit_pipe.joblib")
loaded = joblib.load("credit_pipe.joblib")
p1 = pipe.predict_proba(X_te)[:, 1]
p2 = loaded.predict_proba(X_te)[:, 1]
np.testing.assert_allclose(p1, p2, rtol=0, atol=0)
print(f"保存前後の一致: 最大差 {np.abs(p1 - p2).max():.1e}（完全一致）")

保存前後の一致: 最大差 0.0e+00（完全一致）


In [3]:
# 推論時間の測定：1000件バッチで測り、1件あたりに直す
batch = pd.concat([X_te] * 5, ignore_index=True).head(1000)
t0 = time.perf_counter()
for _ in range(5):
    loaded.predict_proba(batch)
t1 = time.perf_counter()
ms = (t1 - t0) / 5 / len(batch) * 1000
print(f"1000件バッチ: {(t1 - t0) / 5 * 1000:.1f}ms / 1件あたり: {ms:.3f}ms")
# APIと同じく1件ずつ呼ぶ場合を測る（呼び出しごとの固定費がかかる）
one = X_te.head(200)
t0 = time.perf_counter()
for i in range(len(one)):
    loaded.predict_proba(one.iloc[[i]])
t1 = time.perf_counter()
ms_one = (t1 - t0) / len(one) * 1000
print(f"1件ずつ（API相当）: 1件あたり {ms_one:.2f}ms（バッチの約{ms_one / ms:.0f}倍）")
assert np.isfinite(p2).all()

1000件バッチ: 10.0ms / 1件あたり: 0.010ms


1件ずつ（API相当）: 1件あたり 11.77ms（バッチの約1178倍）


## 読み取りどころ

- Pipelineごと固めると保存前後が完全一致すること。
- 1件あたりの推論時間が運用設計（タイムアウト・並列数）の入力になること。バッチと1件ずつ（API相当）では桁が違うこと。
- API化（FastAPI雛形）は本文の抜粋を参照（本ノートは保存・読込・測定まで）。